# 01 — Silver: ED Performance

**Pipeline:** Bronze → Silver  
**Source:** AIHW MyHospitals API — measure MYH0005 (4-hour departure rate), MYH0010, MYH0011, MYH0013  
**Target:** `silver.fact_ed_performance` (Delta table)  

Steps:
1. Read raw JSON landed by ADF from bronze layer
2. Profile schema
3. Filter to WA hospitals
4. Cast types and drop nulls
5. Write as Delta table

In [ ]:
# ------------------------------------------------------------------
# 1. Read raw JSON from bronze layer
#    ADF lands files at: Files/bronze/aihw/measures/{measure_code}/raw.json
# ------------------------------------------------------------------
MEASURE_CODES = ["MYH0005", "MYH0010", "MYH0011", "MYH0013"]

from pyspark.sql.functions import col, lit, explode
from functools import reduce
from pyspark.sql import DataFrame

dfs = []
for code in MEASURE_CODES:
    path = f"Files/bronze/aihw/measures/{code}/raw.json"
    raw = spark.read.option("multiline", "true").json(path)
    # API returns {result: [...], version_information: {...}}
    df = raw.select(explode(col("result")).alias("r")).select("r.*")
    df = df.withColumn("measure_code", lit(code))
    dfs.append(df)
    print(f"{code}: {df.count()} rows loaded")

# Union all measures
combined = reduce(DataFrame.unionByName, dfs, dfs[0])
print(f"\nTotal rows across all measures: {combined.count()}")

In [ ]:
# ------------------------------------------------------------------
# 2. Profile schema — understand the raw structure
# ------------------------------------------------------------------
combined.printSchema()
combined.show(5, truncate=False)

In [ ]:
# ------------------------------------------------------------------
# 3. Filter to WA hospitals and cast types
# ------------------------------------------------------------------
from pyspark.sql.functions import to_date, when

silver_df = (
    combined
    # Keep only hospital-level WA records
    .filter(col("reporting_unit_type.reporting_unit_type_code") == "H")
    .filter(col("state") == "WA")
    # Select and rename key fields
    .select(
        col("reporting_unit_code").alias("hospital_code"),
        col("reporting_unit_name").alias("hospital_name"),
        col("measure_code"),
        col("time_period_start"),
        col("time_period_end"),
        col("value").cast("double").alias("value"),
        col("unit"),
        col("suppressed")
    )
    # Cast time period to date
    .withColumn("time_period_start", to_date(col("time_period_start")))
    .withColumn("time_period_end", to_date(col("time_period_end")))
    # Drop rows missing critical fields
    .dropna(subset=["hospital_code", "value", "time_period_start"])
    # Exclude suppressed values
    .filter((col("suppressed").isNull()) | (col("suppressed") == False))
)

print(f"WA hospital records after filtering: {silver_df.count()}")
silver_df.show(10)

In [ ]:
# ------------------------------------------------------------------
# 4. Data quality check before writing
# ------------------------------------------------------------------
# MYH0005 values should be 0-100 (percentage)
pct_measures = silver_df.filter(col("measure_code") == "MYH0005")
invalid = pct_measures.filter((col("value") < 0) | (col("value") > 100)).count()
print(f"MYH0005 invalid percentage values: {invalid}")

# Check distinct hospitals
print(f"Distinct WA hospitals: {silver_df.select('hospital_code').distinct().count()}")
print(f"Date range: {silver_df.agg({'time_period_start': 'min'}).collect()[0][0]} to {silver_df.agg({'time_period_start': 'max'}).collect()[0][0]}")

In [ ]:
# ------------------------------------------------------------------
# 5. Write silver Delta table
# ------------------------------------------------------------------
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.fact_ed_performance")

print("silver.fact_ed_performance written successfully")
print(f"Final row count: {spark.table('silver.fact_ed_performance').count()}")